In [ ]:
import numpy as np
import cv2
import pandas as pd
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

In [55]:
data = []
labels = []
classes = 43
cur_path = os.getcwd()
img_size = (30, 30, 3)

for i in range(classes):
    path = os.path.join(cur_path, 'archive', 'Train', str(i))
    images = os.listdir(path)
    for img in images:
        try:
            image = cv2.imread(f'archive/Train/{i}/{img}')
            image = cv2.resize(image, (img_size[0], img_size[1]))
            data.append(image)
            labels.append(i)
        except:
            print('Error loading images')

X = np.array(data)
Y = np.array(labels)


In [56]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=.2)

X_train = X_train / 255.0
X_val = X_val / 255.0

encoder = OneHotEncoder(sparse_output=False)
Y_train_reshaped = Y_train.reshape(-1, 1)
Y_val_reshaped = Y_val.reshape(-1, 1)

Y_train = encoder.fit_transform(Y_train_reshaped)
Y_val = encoder.transform(Y_val_reshaped)

In [65]:
input = tf.keras.layers.Input(shape=img_size)
x = tf.keras.layers.Conv2D(32, (5, 5), activation='relu', padding='same')(input)
x = tf.keras.layers.Conv2D(32, (5, 5), activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)

x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPool2D(pool_size=(2, 2))(x)
# x = tf.keras.layers.Dropout(rate=.2)(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
output = tf.keras.layers.Dense(43, activation='softmax')(x)

In [66]:
model = tf.keras.Model(inputs=input, outputs=output)
model.compile(loss=['categorical_crossentropy'], optimizer='adam', metrics=['accuracy'])

In [67]:
model.fit(X_train, Y_train, epochs=5, batch_size=32, validation_data=(X_val, Y_val))

Epoch 1/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 102s 99ms/step - accuracy: 0.4728 - loss: 1.9664 - val_accuracy: 0.9445 - val_loss: 0.2148
Epoch 2/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 104s 106ms/step - accuracy: 0.9693 - loss: 0.1039 - val_accuracy: 0.9756 - val_loss: 0.0813
Epoch 3/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 114s 116ms/step - accuracy: 0.9872 - loss: 0.0446 - val_accuracy: 0.9869 - val_loss: 0.0420
Epoch 4/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 127s 129ms/step - accuracy: 0.9906 - loss: 0.0317 - val_accuracy: 0.9851 - val_loss: 0.0606
Epoch 5/5
981/981 ━━━━━━━━━━━━━━━━━━━━ 145s 148ms/step - accuracy: 0.9912 - loss: 0.0303 - val_accuracy: 0.9851 - val_loss: 0.0547


In [ ]:
test_df = pd.read_csv('archive/Test.csv')
y_test = test_df['ClassId'].values
test_images_paths = test_df['Path'].values

data_test = []
for img_path in test_images_paths:
    try:
        image = cv2.imread(f'archive/{img_path}')
        image = cv2.resize(image, (img_size[0], img_size[1]))
        data_test.append(image)
    except:
        print(f"Error loading test image: {img_path}")
        
X_test = np.array(data_test)
X_test = X_test.astype(np.float32) / 255.0
print(f"Loaded {len(X_test)} test images.")

In [ ]:
pred_probs = model.predict(X_test)
pred_classes = np.argmax(pred_probs, axis=-1)

test_accuracy = accuracy_score(pred_classes, y_test)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
plt.figure(figsize=(12, 8))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(X_test[i])
    plt.title(f"Pred: {pred_classes[i]} | Actual: {y_test[i]}")
    plt.axis('off')
plt.show()